# LiteRT Export Demo — Gemma 3 270M

Verified with `keras` + `keras-hub` + `litert-torch` master.

**Note:** Switching `KERAS_BACKEND` (TF ↔ Torch) requires a **runtime restart**.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import keras_hub

PRESET = "hf://google/gemma-3-270m-it"
SEQ_LEN = 128

model = keras_hub.models.Gemma3CausalLM.from_preset(PRESET)
model.preprocessor.sequence_length = SEQ_LEN
model.generate(["What is Keras?"], max_length=16)

## 1. TF Backend Export

In [ ]:
model.export("gemma3_270m_tf.tflite", format="litert")
print("Size (MB):", round(os.path.getsize("gemma3_270m_tf.tflite") / 1e6, 1))

## 2. Torch Backend Export

**Restart runtime, then run:**

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import keras_hub
from keras import layers

PRESET = "hf://google/gemma-3-270m-it"
SEQ_LEN = 128

model = keras_hub.models.Gemma3CausalLM.from_preset(PRESET)
model.preprocessor.sequence_length = SEQ_LEN

processed = model.preprocessor({"prompts": ["What is Keras?"], "responses": [""]})
model(processed[0])

input_signature = [{
    "token_ids": layers.InputSpec(dtype="int32", shape=(1, SEQ_LEN)),
    "padding_mask": layers.InputSpec(dtype="int32", shape=(1, SEQ_LEN)),
}]

model.export("gemma3_270m_torch.tflite", format="litert", input_signature=input_signature)
print("Size (MB):", round(os.path.getsize("gemma3_270m_torch.tflite") / 1e6, 1))

## 3. Built-in FP16 Quantization

**Restart runtime with TF backend, then run:**

In [ ]:
import os, tensorflow as tf, keras_hub
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

model = keras_hub.models.Gemma3CausalLM.from_preset("hf://google/gemma-3-270m-it")
model.preprocessor.sequence_length = 128
model.generate(["Hi"], max_length=4)

model.export(
    "gemma3_270m_fp16.tflite",
    format="litert",
    optimizations=[tf.lite.Optimize.DEFAULT],
)
print("FP16 size (MB):", round(os.path.getsize("gemma3_270m_fp16.tflite") / 1e6, 1))

## 4. Mixed-Precision via ai-edge-quantizer

Post-export on the FP32 `.tflite` from Step 1.

In [ ]:
from ai_edge_quantizer import quantizer, recipe

qt = quantizer.Quantizer("gemma3_270m_tf.tflite")
qt.load_quantization_recipe(recipe.dynamic_wi8_afp32())

qt.update_quantization_recipe([
    {"op_type": "FULLY_CONNECTED", "scope": ".*attention.*", "algorithm_key": "no_quantize"},
    {"op_type": "FULLY_CONNECTED", "scope": ".*feedforward.*", "algorithm_key": "dynamic_wi4_afp32"},
])

qt.quantize().export_model("gemma3_270m_mixed.tflite")
print("Mixed size (MB):", round(os.path.getsize("gemma3_270m_mixed.tflite") / 1e6, 1))

## 5. Verify Inference

In [ ]:
import numpy as np
from ai_edge_litert.interpreter import Interpreter

runner = Interpreter("gemma3_270m_mixed.tflite").get_signature_runner("serving_default")
result = runner(
    args_0=np.ones((1, 128), dtype=np.int32),
    args_0_1=np.ones((1, 128), dtype=np.int32),
)
print("Output shape:", result["output_0"].shape)